# AIMv2's Autoregressive Bet Against CLIP Orthodoxy

**Paper:** [https://arxiv.org/abs/2411.14402](https://arxiv.org/abs/2411.14402)  
**Authors:** Enrico Fini, Mustafa Shukor, Xiujun Li, Philipp Dufter, Michal Klein, David Haldimann, Sai Aitharaju, Victor Guilherme Turrisi da Costa, Louis Béthune, Zhe Gan, Alexander T Toshev, Marcin Eichner, Moin Nabi, Yinfei Yang, Joshua M. Susskind, Alaaeldin El-Nouby  
**Repository:** [https://github.com/apple/ml-aim](https://github.com/apple/ml-aim)  
**Framework:** pytorch  
**License:** NOASSERTION  

---

*Reproduction generated by Vivory Research — runs on free-tier hardware (Kaggle T4 / Oracle CPU / GitHub Actions).*
*Produced: 2026-05-06 13:18 UTC*


## 1. Setup

Install dependencies from the paper's `requirements.txt`. Some packages may need GPU-specific wheels — adjust for your Colab/Kaggle runtime.

In [ ]:
!pip install --quiet --upgrade pip
!pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


## 2. Repository

Clone the reference implementation.

In [ ]:
!git clone --depth 1 https://github.com/apple/ml-aim
%cd ml-aim
!ls -la


## 3. Dataset

Download the dataset. Replace this cell with the dataset-specific loading code from the repository's README or `scripts/download_data.sh`.

In [ ]:
!pip install --quiet datasets
from datasets import load_dataset
ds = load_dataset("huggingface/badges")
print(ds)


## 4. Configuration

Core hyperparameters. Consider reducing epochs/batch size to fit free-tier GPU limits (Kaggle T4: 16GB VRAM, 30h/week; Colab: variable).

In [ ]:
import os, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Reduced for free-tier — adjust if you have more GPU budget.
CONFIG = {
    "seed": SEED,
    "max_epochs": 1,
    "batch_size": 16,
    "learning_rate": 1e-4,
    "subset_fraction": 0.1,  # use 10% of data for quick reproduction
}
print(json.dumps(CONFIG, indent=2))


## 5+6. Paper-aware evaluation (auto-generated)

The cell below was generated by Vivory's reproduction agent (Opus 4.7) from the paper's abstract, body, repo README, and claimed_metrics. It performs real measurement on a small subset and writes the result to `/kaggle/working/metrics.json` for the runner to ingest.

In [ ]:
!pip install -q transformers accelerate datasets

import os, json, torch
import torch.nn.functional as F
from transformers import AutoModel, AutoProcessor
from datasets import load_dataset

os.makedirs("/kaggle/working", exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

def write_metrics(m):
    with open("/kaggle/working/metrics.json", "w") as f:
        json.dump(m, f)
    print(m)

try:
    # AIMv2 LiT variant supports zero-shot classification (CLIP-like API)
    model_id = "apple/aimv2-large-patch14-224-lit"
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        model_id, trust_remote_code=True, torch_dtype=torch.float16
    ).to(device).eval()

    # Public ImageNet-derived dataset (Imagenette: 10 ImageNet-1k classes)
    ds = None
    for cfg in ["full_size", "320px", "160px"]:
        try:
            ds = load_dataset("frgfm/imagenette", cfg, split="validation")
            print(f"Loaded frgfm/imagenette {cfg}: {len(ds)} samples")
            break
        except Exception as e:
            print(f"cfg {cfg} failed: {e}")
    if ds is None:
        ds = load_dataset("frgfm/imagenette", split="validation")

    wnid_to_name = {
        "n01440764": "tench", "n02102040": "English springer spaniel",
        "n02979186": "cassette player", "n03000684": "chainsaw",
        "n03028079": "church", "n03394916": "French horn",
        "n03417042": "garbage truck", "n03425413": "gas pump",
        "n03445777": "golf ball", "n03888257": "parachute",
    }
    label_feat = ds.features["label"]
    raw_names = getattr(label_feat, "names", None)
    if raw_names and isinstance(raw_names[0], str) and raw_names[0].startswith("n"):
        class_names = [wnid_to_name.get(c, c) for c in raw_names]
    elif raw_names:
        class_names = list(raw_names)
    else:
        class_names = list(wnid_to_name.values())
    print("classes:", class_names)

    text_prompts = [f"a photo of a {c}" for c in class_names]

    # Encode text inputs once
    text_full = processor(text=text_prompts, return_tensors="pt", padding=True)
    text_kwargs = {k: v.to(device) for k, v in text_full.items()
                   if k in ("input_ids", "attention_mask")}

    N = min(300, len(ds))
    ds_sub = ds.shuffle(seed=42).select(range(N))

    correct = 0
    total = 0
    for i, ex in enumerate(ds_sub):
        img = ex["image"]
        if hasattr(img, "convert"):
            img = img.convert("RGB")
        label = ex["label"]
        try:
            img_inputs = processor(images=img, return_tensors="pt")
            img_kwargs = {k: v.to(device) for k, v in img_inputs.items()
                          if k == "pixel_values"}
            with torch.no_grad():
                outputs = model(**img_kwargs, **text_kwargs)
                if hasattr(outputs, "logits_per_image"):
                    logits = outputs.logits_per_image
                else:
                    # Manual cosine similarity fallback
                    img_emb = outputs.image_embeds if hasattr(outputs, "image_embeds") else outputs[0]
                    txt_emb = outputs.text_embeds if hasattr(outputs, "text_embeds") else outputs[1]
                    img_emb = F.normalize(img_emb.float(), dim=-1)
                    txt_emb = F.normalize(txt_emb.float(), dim=-1)
                    logits = img_emb @ txt_emb.T
                pred = int(logits.argmax(dim=-1).item())
        except Exception as e:
            if i < 3:
                print(f"sample {i} err: {e}")
            continue
        if pred == label:
            correct += 1
        total += 1
        if (i + 1) % 50 == 0:
            print(f"{i+1}/{N}  running acc={correct/max(total,1):.4f}")

    if total < 10:
        print(f"Too few successful samples ({total}); marking unsupported")
        write_metrics({"unsupported_infrastructure": 1.0})
    else:
        acc = correct / total * 100.0
        print(f"correct={correct}, total={total}, acc={acc:.2f}")
        write_metrics({"accuracy_imagenet": float(acc)})

except Exception as e:
    import traceback; traceback.print_exc()
    print(f"Top-level failure: {e}")
    write_metrics({"unsupported_infrastructure": 1.0})

## Appendix — Reproduction policy

This notebook runs on **free-tier hardware only**:

- **Kaggle Notebooks** — T4 GPU, 30h/week quota
- **Oracle Cloud** — ARM 4-core CPU, no GPU
- **GitHub Actions** — 2-core CPU, no GPU, 6h timeout
- **Colab** — variable T4/V100, 12h sessions (manual only)

If the full experiment exceeds these limits, reduce `max_epochs` / `subset_fraction` in the config cell and note the delta in the reproduction report.
